# SQL: the data preparation, query by query

The pipeline in this project runs on pandas. This notebook redoes the main preparation and summary steps in SQL (DuckDB dialect, the same SQL that runs on warehouse engines like Snowflake or BigQuery), directly on the processed CSVs. The queries live in the `sql/` folder and are executed here as-is.

In [1]:
from pathlib import Path
import duckdb
from IPython.display import display

con = duckdb.connect()

def run(name):
    sql = Path(f'../sql/{name}').read_text(encoding='utf-8')
    print(sql)
    return con.execute(sql).df()

print('DuckDB', duckdb.__version__)

DuckDB 1.5.4


## 1. Annual failures by treatment group
A join between the annual panel and the treatment table, pivoted with conditional aggregation.

In [2]:
display(run('01_failures_by_year_and_group.sql'))

-- Annual PME failures by treatment group.
-- Joins the annual panel with the sector treatment table and pivots the
-- clean sample (baseline 2015-2019, post 2023+) into one row per year.
SELECT
    p.year,
    SUM(CASE WHEN t.treated THEN p.failures END)      AS failures_treated,
    SUM(CASE WHEN NOT t.treated THEN p.failures END)  AS failures_control,
    ROUND(SUM(p.failures))                            AS failures_total
FROM read_csv('../data/processed/sector_panel_annual.csv') AS p
JOIN read_csv('../data/processed/treatment.csv') AS t
  ON p.sector = t.sector
WHERE p.in_sample
GROUP BY p.year
ORDER BY p.year;



,year,failures_treated,failures_control,failures_total
0,2015,35834.0,23591.0,59425.0
1,2016,33518.0,20976.0,54494.0
2,2017,32084.0,19221.0,51305.0
3,2018,31813.0,19137.0,50950.0
4,2019,30334.0,17885.0,48219.0
5,2023,33121.0,19756.0,52877.0
6,2024,37701.0,24048.0,61749.0
7,2025,40136.0,24057.0,64193.0


## 2. The 2x2 difference-in-differences table
A CTE computes the four group-by-period means, the outer query lines them up and derives the raw DiD in one pass. The result matches the decomposition in notebook 03.

In [3]:
display(run('02_did_group_means.sql'))

-- The 2x2 difference-in-differences table in one query.
-- Averages log(1 + failures) by treatment group and period, then the outer
-- query lines up the four cells and computes the raw DiD estimate.
WITH cells AS (
    SELECT
        t.treated,
        p.post,
        AVG(LN(1 + p.failures)) AS mean_log_failures
    FROM read_csv('../data/processed/sector_panel_annual.csv') AS p
    JOIN read_csv('../data/processed/treatment.csv') AS t
      ON p.sector = t.sector
    WHERE p.in_sample
    GROUP BY t.treated, p.post
)
SELECT
    MAX(CASE WHEN treated AND post = 0 THEN mean_log_failures END)     AS treated_before,
    MAX(CASE WHEN treated AND post = 1 THEN mean_log_failures END)     AS treated_after,
    MAX(CASE WHEN NOT treated AND post = 0 THEN mean_log_failures END) AS control_before,
    MAX(CASE WHEN NOT treated AND post = 1 THEN mean_log_failures END) AS control_after,
    (MAX(CASE WHEN treated AND post = 1 THEN mean_log_failures END)
     - MAX(CASE WHEN treated AND post = 0

,treated_before,treated_after,control_before,control_after,did_raw
0,8.58352,8.695018,8.124977,8.339058,-0.102582


## 3. Post-hike growth ranking by sector
Window functions: a partitioned average builds each sector's baseline, RANK orders sectors by their relative increase.

In [4]:
display(run('03_sector_growth_ranking.sql'))

-- Post-hike failure growth by sector, against its own baseline.
-- Window functions: the baseline mean is computed per sector with a
-- partitioned AVG, then sectors are ranked by their relative increase.
WITH annual AS (
    SELECT
        p.sector,
        t.label,
        t.bank_dependence,
        p.year,
        p.failures,
        AVG(CASE WHEN p.in_baseline THEN p.failures END)
            OVER (PARTITION BY p.sector) AS baseline_mean
    FROM read_csv('../data/processed/sector_panel_annual.csv') AS p
    JOIN read_csv('../data/processed/treatment.csv') AS t
      ON p.sector = t.sector
    WHERE p.in_sample
)
SELECT
    sector,
    label,
    bank_dependence,
    ROUND(AVG(failures), 0)                                   AS post_mean,
    ROUND(ANY_VALUE(baseline_mean), 0)                        AS baseline_mean,
    ROUND(100.0 * (AVG(failures) / ANY_VALUE(baseline_mean) - 1), 1) AS growth_pct,
    RANK() OVER (ORDER BY AVG(failures) / ANY_VALUE(baseline_mean) DESC) AS rank_by

,sector,label,bank_dependence,post_mean,baseline_mean,growth_pct,rank_by_growth
0,H,Transport & storage,27.733333,2823.0,1893.0,49.1,1
1,JZ,Information & communication,31.700000,1932.0,1357.0,42.3,2
2,MN,Business services,61.266667,7667.0,6109.0,25.5,3
3,I,Accommodation & food,71.700000,8563.0,7526.0,13.8,4
4,PS,"Education, health, personal services",68.833333,6054.0,5455.0,11.0,5
5,G,Trade & auto repair,60.033333,13229.0,12195.0,8.5,6
6,FZ,Construction,59.366667,13681.0,12935.0,5.8,7
7,BE,Industry,38.450000,4185.0,3977.0,5.2,8
8,AZ,Agriculture,80.600000,1474.0,1433.0,2.8,9


## 4. Department profile: ZFRR intensity and failure burden
A three-way join (panel, ZFRR intensity, FLORES firm counts) producing a failure rate per 1,000 firms.

In [5]:
display(run('04_departments_zfrr_profile.sql'))

-- Department profile: ZFRR intensity, firm stock and failure burden.
-- Three-way join between the department panel, the ZFRR intensity table and
-- the FLORES firm counts, with a failure rate per 1,000 firms.
WITH post AS (
    SELECT
        dept,
        ANY_VALUE(zfrr_share)  AS zfrr_share,
        AVG(failures)          AS avg_failures_post
    FROM read_csv('../data/processed/dept_panel_annual.csv', types = {'dept': 'VARCHAR'})
    WHERE post = 1 AND failures > 0
    GROUP BY dept
)
SELECT
    p.dept,
    ROUND(p.zfrr_share, 2)                                  AS zfrr_share,
    f.firm_count,
    ROUND(p.avg_failures_post, 0)                           AS avg_failures_post,
    ROUND(1000.0 * p.avg_failures_post / f.firm_count, 2)   AS failures_per_1000_firms
FROM post AS p
JOIN read_csv('../data/processed/zfrr_intensity.csv', types = {'dept': 'VARCHAR'}) AS z
  ON p.dept = z.dept
JOIN read_csv('../data/processed/dept_firm_counts.csv', types = {'dept': 'VARCHAR'}) AS f
  ON p.dep

,dept,zfrr_share,firm_count,avg_failures_post,failures_per_1000_firms
0,77,0.09,84448.0,2148.0,25.44
1,93,0.00,104600.0,2357.0,22.53
2,972,0.00,26434.0,478.0,18.08
3,974,0.13,58126.0,1030.0,17.72
4,87,0.65,22788.0,385.0,16.88
5,59,0.08,141398.0,2344.0,16.58
6,66,0.61,36820.0,603.0,16.39
7,13,0.05,160338.0,2547.0,15.89
8,91,0.00,72160.0,1130.0,15.66
9,33,0.50,126700.0,1950.0,15.39
